# Hypothesis Testing

## Learning Objectives
1. Implement a two-sample t-test from scratch and verify against scipy
2. Perform power analysis to compute required sample size for A/B tests
3. Detect and correct for multiple testing (Bonferroni and Benjamini-Hochberg FDR)
4. Compare t-test vs Mann-Whitney U vs permutation test under non-normality

In [ ]:
import numpy as np
import scipy.stats as stats
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
from statsmodels.stats.power import TTestIndPower
import matplotlib.pyplot as plt
from itertools import combinations

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Imports OK')
print('NumPy:', np.__version__)

## Level 1: Two-Sample t-Test from Scratch

The t-statistic for comparing two independent groups:
  t = (x_bar_A - x_bar_B) / sqrt(s_A^2/n_A + s_B^2/n_B)

Under H0 (no difference), t follows a t-distribution with Welch's df.
p-value = 2 * P(T >= |t| | H0) for two-tailed test.

In [ ]:
np.random.seed(42)

# --- Generate two groups ---
# Group A: control, conversion rate 10%
# Group B: treatment, conversion rate 12.5%
true_mu_A = 0.10
true_mu_B = 0.125
n_A, n_B  = 300, 300
sigma     = np.sqrt(true_mu_A * (1 - true_mu_A))  # Bernoulli std

# Continuous metric (revenue per user, for t-test applicability)
# Approximate as Gaussian for simplicity
A = np.random.normal(true_mu_A, sigma, n_A)
B = np.random.normal(true_mu_B, sigma, n_B)

print(f'Group A: n={n_A}, mean={A.mean():.4f}, std={A.std():.4f}')
print(f'Group B: n={n_B}, mean={B.mean():.4f}, std={B.std():.4f}')
print(f'Observed difference: {B.mean() - A.mean():.4f}')

# --- t-test from scratch (Welch: unequal variance) ---
def welch_ttest_from_scratch(x, y):
    n1, n2 = len(x), len(y)
    mean1, mean2 = x.mean(), y.mean()
    var1, var2 = x.var(ddof=1), y.var(ddof=1)  # unbiased variance

    # Standard error of the difference
    se = np.sqrt(var1/n1 + var2/n2)

    # t statistic
    t_stat = (mean1 - mean2) / se

    # Welch-Satterthwaite degrees of freedom
    df = (var1/n1 + var2/n2)**2 / ((var1/n1)**2/(n1-1) + (var2/n2)**2/(n2-1))

    # p-value: two-tailed
    p_value = 2 * stats.t.sf(abs(t_stat), df=df)

    # 95% confidence interval for the difference
    diff = mean1 - mean2
    ci_lo = diff - stats.t.ppf(0.975, df) * se
    ci_hi = diff + stats.t.ppf(0.975, df) * se

    return t_stat, p_value, df, ci_lo, ci_hi

t_s, p_s, df_s, ci_lo, ci_hi = welch_ttest_from_scratch(A, B)
print(f'\nOur t-test: t={t_s:.4f}, p={p_s:.4f}, df={df_s:.1f}')
print(f'  95% CI for (A-B): [{ci_lo:.4f}, {ci_hi:.4f}]')

# Verify against scipy
t_scipy, p_scipy = ttest_ind(A, B, equal_var=False)  # Welch
print(f'scipy ttest_ind:  t={t_scipy:.4f}, p={p_scipy:.4f}')
print(f'Agreement: t diff={abs(t_s-t_scipy):.2e}, p diff={abs(p_s-p_scipy):.2e}')

# Decision
alpha = 0.05
print(f'\nDecision (alpha={alpha}): {"Reject H0 (significant)" if p_scipy < alpha else "Fail to reject H0"}')

# --- Cohen's d effect size ---
pooled_std = np.sqrt(((n_A-1)*A.var(ddof=1) + (n_B-1)*B.var(ddof=1)) / (n_A+n_B-2))
cohens_d   = (B.mean() - A.mean()) / pooled_std
print(f"Cohen's d = {cohens_d:.4f}  ('small': 0.2, 'medium': 0.5, 'large': 0.8)")

# --- Visualize ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bins = np.linspace(-0.6, 0.8, 50)
axes[0].hist(A, bins=bins, alpha=0.6, density=True, label=f'A (mean={A.mean():.3f})', color='blue')
axes[0].hist(B, bins=bins, alpha=0.6, density=True, label=f'B (mean={B.mean():.3f})', color='red')
axes[0].axvline(A.mean(), color='blue', ls='--', lw=2)
axes[0].axvline(B.mean(), color='red',  ls='--', lw=2)
axes[0].set_xlabel('Metric value'); axes[0].set_ylabel('Density')
axes[0].set_title(f'Two-Sample t-Test\nt={t_scipy:.3f}, p={p_scipy:.4f}', fontweight='bold')
axes[0].legend(fontsize=9)

# t-distribution with shaded p-value region
df_plot = min(df_s, 200)
t_range = np.linspace(-4, 4, 300)
t_pdf   = stats.t.pdf(t_range, df=df_plot)
axes[1].plot(t_range, t_pdf, 'k-', lw=2, label=f't-distribution df={df_plot:.0f}')
axes[1].fill_between(t_range, 0, t_pdf, where=(np.abs(t_range) >= abs(t_scipy)),
                     alpha=0.4, color='red', label=f'p-value region={p_scipy:.4f}')
axes[1].axvline( abs(t_scipy), color='red', ls='--', lw=1.5)
axes[1].axvline(-abs(t_scipy), color='red', ls='--', lw=1.5)
axes[1].set_xlabel('t statistic'); axes[1].set_ylabel('Density')
axes[1].set_title(f'p-value = shaded area (two-tailed)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_05_ttest.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_05_ttest.png')

## Level 2: Power Analysis -- Computing Required Sample Size

Power = P(reject H0 | H1 true) = 1 - beta.

For a two-sample t-test, required n per group:
  n = 2 * sigma^2 * (z_{alpha/2} + z_beta)^2 / delta^2

where delta is the minimum detectable effect (MDE).
Always compute required n BEFORE collecting data.

In [ ]:
# --- Power analysis parameters ---
alpha  = 0.05   # significance level (Type I error rate)
power  = 0.80   # desired power (1 - beta = 0.80 means 20% Type II error)
sigma  = 0.30   # estimated std of metric (prior knowledge or pilot data)

# MDEs: what is the smallest effect we care about detecting?
mde_values = [0.01, 0.02, 0.05, 0.10, 0.20]   # absolute differences

# --- Formula-based n calculation ---
z_alpha2 = stats.norm.ppf(1 - alpha/2)   # z for alpha/2 = 1.96 for alpha=0.05
z_beta   = stats.norm.ppf(power)          # z for 1-beta = 0.84 for power=0.80

print(f'Power analysis parameters:')
print(f'  alpha={alpha} (z={z_alpha2:.3f})')
print(f'  power={power} (z_beta={z_beta:.3f})')
print(f'  sigma={sigma}')
print()
print(f'  {'MDE':>8} {'n/group':>10} {'Total n':>10} {'Cohen d':>10}')
print('  ' + '-' * 42)

req_ns = []
for mde in mde_values:
    # Formula: n = 2 sigma^2 (z_a2 + z_b)^2 / mde^2
    n_req = int(np.ceil(2 * sigma**2 * (z_alpha2 + z_beta)**2 / mde**2))
    d_cohen = mde / sigma
    req_ns.append(n_req)
    print(f'  {mde:>8.2f} {n_req:>10} {2*n_req:>10} {d_cohen:>10.3f}')

# --- Verify with statsmodels TTestIndPower ---
print('\nVerification with statsmodels TTestIndPower:')
analysis = TTestIndPower()
for mde in [0.05, 0.10]:
    d_cohen = mde / sigma
    n_sm = analysis.solve_power(effect_size=d_cohen, alpha=alpha, power=power, ratio=1.0)
    print(f'  MDE={mde}: formula n={int(np.ceil(2*sigma**2*(z_alpha2+z_beta)**2/mde**2))}',
          f'statsmodels n={int(np.ceil(n_sm))}')

# --- Power curves: power as a function of n ---
n_range = np.arange(50, 2000, 10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Power vs n for different MDEs
for mde in [0.02, 0.05, 0.10]:
    d_c = mde / sigma
    powers = [analysis.solve_power(effect_size=d_c, alpha=alpha, nobs1=n, ratio=1.0)
              for n in n_range]
    axes[0].plot(n_range, powers, lw=2, label=f'MDE={mde} (d={d_c:.2f})')

axes[0].axhline(0.80, color='gray', ls='--', label='80% power target')
axes[0].axhline(0.95, color='lightgray', ls=':', label='95% power target')
axes[0].set_xlabel('n per group')
axes[0].set_ylabel('Statistical power')
axes[0].set_title('Power Curves: How n Affects Detection Ability', fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=9)

# Required n vs MDE
mde_range = np.linspace(0.01, 0.25, 300)
n_required = np.ceil(2 * sigma**2 * (z_alpha2 + z_beta)**2 / mde_range**2)
axes[1].plot(mde_range, n_required, 'b-', lw=2)
axes[1].scatter(mde_values, req_ns, color='red', s=80, zorder=5, label='Annotated MDEs')
axes[1].set_xlabel('Minimum Detectable Effect (MDE)')
axes[1].set_ylabel('Required n per group')
axes[1].set_title('Required Sample Size vs Effect Size\n(alpha=0.05, power=0.80)', fontweight='bold')
axes[1].set_ylim(0, 3000)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stats_05_power.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_05_power.png')

# --- Empirical power verification ---
print('\nEmpirical power check (10,000 simulations):')
n_sims = 10_000
for mde in [0.05, 0.10]:
    n_each = int(np.ceil(2 * sigma**2 * (z_alpha2 + z_beta)**2 / mde**2))
    rejected = 0
    for _ in range(n_sims):
        ga = np.random.normal(0.0, sigma, n_each)
        gb = np.random.normal(mde, sigma, n_each)
        _, p = ttest_ind(ga, gb, equal_var=False)
        if p < alpha:
            rejected += 1
    emp_power = rejected / n_sims
    print(f'  MDE={mde}, n={n_each}: theoretical power=0.80, empirical={emp_power:.3f}')

## Real-World Example 1: A/B Test Analysis (Complete Pipeline)

Full A/B test analysis including:
- Sample ratio mismatch check
- t-test for means, z-test for proportions
- Effect size (Cohen's d) and practical significance
- Confidence interval interpretation

In [ ]:
np.random.seed(42)

# --- A/B test: website redesign impact on session duration ---
# Control (A): old design, Treatment (B): new design
# Primary metric: session duration in minutes (right-skewed; use log-normal)
n_A, n_B = 1500, 1503  # slight imbalance (normal in practice)

# Simulate log-normal session duration (right-skewed like real web traffic)
# Log-scale parameters: mu=2.5 (exp=12.2 min), sigma=1.0
log_mu_A = 2.5
log_mu_B = 2.58   # slight improvement
log_sig  = 1.0

session_A = np.random.lognormal(log_mu_A, log_sig, n_A)
session_B = np.random.lognormal(log_mu_B, log_sig, n_B)

print(f'Group A: n={n_A}, mean={session_A.mean():.2f} min, median={np.median(session_A):.2f} min')
print(f'Group B: n={n_B}, mean={session_B.mean():.2f} min, median={np.median(session_B):.2f} min')
print(f'Relative lift: {(session_B.mean()-session_A.mean())/session_A.mean()*100:.1f}%')

# --- Sample ratio mismatch check ---
# We expected 50/50 split. Check with chi-square test on assignment.
expected_n = (n_A + n_B) / 2
chi2_srm = (n_A - expected_n)**2/expected_n + (n_B - expected_n)**2/expected_n
p_srm = 1 - stats.chi2.cdf(chi2_srm, df=1)
print(f'\nSample Ratio Mismatch check: chi2={chi2_srm:.4f}, p={p_srm:.4f}')
print(f'SRM detected? {p_srm < 0.01}  (alert if p < 0.01)')

# --- Test 1: t-test on raw values ---
t_raw, p_raw = ttest_ind(session_A, session_B, equal_var=False)
print(f'\nt-test (raw): t={t_raw:.3f}, p={p_raw:.4f}')

# --- Test 2: t-test on log-transformed values (more appropriate for log-normal) ---
log_A = np.log(session_A)
log_B = np.log(session_B)
t_log, p_log = ttest_ind(log_A, log_B, equal_var=False)
print(f't-test (log): t={t_log:.3f}, p={p_log:.4f}')

# --- Effect size ---
pooled_std_log = np.sqrt(((n_A-1)*log_A.var(ddof=1) + (n_B-1)*log_B.var(ddof=1)) / (n_A+n_B-2))
cohens_d_log   = (log_B.mean() - log_A.mean()) / pooled_std_log
print(f"Cohen's d (log scale): {cohens_d_log:.4f}")
print(f'Practical significance: {"small (<0.2)" if abs(cohens_d_log)<0.2 else "medium (0.2-0.5)" if abs(cohens_d_log)<0.5 else "large"}')

# --- Confidence interval for relative lift ---
n_boot = 5000
boot_lifts = []
for _ in range(n_boot):
    sA = np.random.choice(session_A, n_A, replace=True)
    sB = np.random.choice(session_B, n_B, replace=True)
    boot_lifts.append((sB.mean() - sA.mean()) / sA.mean() * 100)

ci_lo_boot = np.percentile(boot_lifts, 2.5)
ci_hi_boot = np.percentile(boot_lifts, 97.5)
print(f'\nBootstrap 95% CI for relative lift: [{ci_lo_boot:.2f}%, {ci_hi_boot:.2f}%]')

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution comparison
cap = np.percentile(np.concatenate([session_A, session_B]), 99)  # cap outliers for plot
bins = np.linspace(0, cap, 50)
axes[0].hist(session_A[session_A < cap], bins=bins, alpha=0.6, density=True, color='blue', label='Control A')
axes[0].hist(session_B[session_B < cap], bins=bins, alpha=0.6, density=True, color='red',  label='Treatment B')
axes[0].axvline(session_A.mean(), color='blue', ls='--', lw=2, label=f'Mean A={session_A.mean():.1f}')
axes[0].axvline(session_B.mean(), color='red',  ls='--', lw=2, label=f'Mean B={session_B.mean():.1f}')
axes[0].set_xlabel('Session duration (min)')
axes[0].set_title(f'A/B Test: Session Duration\np={p_log:.4f} (log-scale t-test)', fontweight='bold')
axes[0].legend(fontsize=8)

# Bootstrap CI distribution
axes[1].hist(boot_lifts, bins=50, density=True, color='purple', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', ls='--', lw=2, label='No lift')
axes[1].fill_betweenx([0, 0.3], ci_lo_boot, ci_hi_boot, alpha=0.2, color='green')
axes[1].axvline(ci_lo_boot, color='green', ls=':', lw=1.5, label=f'95% CI [{ci_lo_boot:.1f}%, {ci_hi_boot:.1f}%]')
axes[1].axvline(ci_hi_boot, color='green', ls=':', lw=1.5)
axes[1].set_xlabel('Relative lift (%)')
axes[1].set_title('Bootstrap CI for Relative Lift', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_05_ab_test.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_05_ab_test.png')

## Real-World Example 2: Multiple Testing Correction

Running 20 simultaneous A/B tests at alpha=0.05 yields ~64% chance of
at least one false positive even if nothing works.

We demonstrate Bonferroni and Benjamini-Hochberg FDR corrections.

In [ ]:
np.random.seed(42)

# --- Simulate 20 metrics, only 3 truly have effects ---
n_tests  = 20
n_each   = 300
alpha    = 0.05
true_effects = [0.0] * 17 + [0.15, 0.20, 0.10]  # 17 nulls, 3 real

p_values_raw = []
for i, true_effect in enumerate(true_effects):
    A = np.random.normal(0.0, 1.0, n_each)
    B = np.random.normal(true_effect, 1.0, n_each)
    _, p = ttest_ind(A, B, equal_var=False)
    p_values_raw.append(p)

p_arr = np.array(p_values_raw)
significant_raw = p_arr < alpha

print(f'Raw p-values (n_tests={n_tests}, alpha={alpha}):')
print(f'  True effects at tests 18-20')
print(f'  Rejections without correction: {significant_raw.sum()}'
      f' ({significant_raw.sum()} total, should be ~3 + noise)')

# Count false positives and true positives
truly_null = [e == 0.0 for e in true_effects]
fp_raw = sum(1 for i, rej in enumerate(significant_raw) if rej and truly_null[i])
tp_raw = sum(1 for i, rej in enumerate(significant_raw) if rej and not truly_null[i])
print(f'  False positives: {fp_raw}, True positives: {tp_raw}')

# --- Bonferroni correction ---
alpha_bonf = alpha / n_tests
significant_bonf = p_arr < alpha_bonf
fp_bonf = sum(1 for i, rej in enumerate(significant_bonf) if rej and truly_null[i])
tp_bonf = sum(1 for i, rej in enumerate(significant_bonf) if rej and not truly_null[i])
print(f'\nBonferroni (alpha_corrected={alpha_bonf:.4f}):')
print(f'  Rejections: {significant_bonf.sum()}, FP={fp_bonf}, TP={tp_bonf}')

# --- Benjamini-Hochberg FDR ---
def bh_correction(p_vals, fdr=0.05):
    # Sort p-values, compute BH thresholds
    m = len(p_vals)
    sorted_idx = np.argsort(p_vals)
    sorted_p   = p_vals[sorted_idx]
    # BH threshold for rank k: k/m * fdr
    bh_thresholds = (np.arange(1, m+1) / m) * fdr
    # Reject all p_k <= threshold up to the largest k that satisfies the condition
    reject = np.zeros(m, dtype=bool)
    max_k = -1
    for k in range(m-1, -1, -1):
        if sorted_p[k] <= bh_thresholds[k]:
            max_k = k
            break
    if max_k >= 0:
        reject[:max_k+1] = True
    # Map back to original order
    result = np.zeros(m, dtype=bool)
    result[sorted_idx[:max_k+1]] = True
    return result, bh_thresholds

significant_bh, bh_thresh = bh_correction(p_arr, fdr=alpha)
fp_bh = sum(1 for i, rej in enumerate(significant_bh) if rej and truly_null[i])
tp_bh = sum(1 for i, rej in enumerate(significant_bh) if rej and not truly_null[i])
print(f'\nBenjamini-Hochberg FDR (q={alpha}):')
print(f'  Rejections: {significant_bh.sum()}, FP={fp_bh}, TP={tp_bh}')

# --- Comparison across many simulations ---
print('\nSimulation: empirical FP rate over 1,000 experiments')
n_sims_mult = 1000
fp_rates = {'Raw': [], 'Bonferroni': [], 'BH-FDR': []}
for _ in range(n_sims_mult):
    ps = np.array([ttest_ind(np.random.randn(n_each), np.random.randn(n_each), equal_var=False)[1]
                   for _ in range(n_tests)])
    fp_rates['Raw'].append(np.any(ps < alpha))
    fp_rates['Bonferroni'].append(np.any(ps < alpha/n_tests))
    rej_bh, _ = bh_correction(ps, fdr=alpha)
    fp_rates['BH-FDR'].append(np.any(rej_bh))

print(f'  (All null hypotheses true, so all rejections are FPs)')
for method, rates in fp_rates.items():
    print(f'  {method:<15}: FWER = {np.mean(rates):.3f}  (target for Bonferroni/BH: <= {alpha})')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sorted_idx_plot = np.argsort(p_arr)
sorted_p_plot   = p_arr[sorted_idx_plot]
ranks = np.arange(1, n_tests + 1)

axes[0].scatter(ranks, sorted_p_plot, color='blue', s=60, zorder=5, label='p-values (sorted)')
axes[0].axhline(alpha, color='green', ls='--', lw=1.5, label=f'alpha={alpha} (no correction)')
axes[0].axhline(alpha/n_tests, color='red', ls='--', lw=1.5, label=f'Bonferroni={alpha/n_tests:.4f}')
axes[0].plot(ranks, bh_thresh, 'purple', ls='--', lw=1.5, label='BH thresholds')
axes[0].set_xlabel('Rank (sorted by p-value)')
axes[0].set_ylabel('p-value')
axes[0].set_title('Multiple Testing: Correction Methods', fontweight='bold')
axes[0].legend(fontsize=8)

# Bar chart of FP rates
methods = list(fp_rates.keys())
fwer_vals = [np.mean(fp_rates[m]) for m in methods]
colors = ['red', 'green', 'blue']
bars = axes[1].bar(methods, fwer_vals, color=colors, alpha=0.7)
axes[1].axhline(alpha, color='black', ls='--', lw=2, label=f'Target alpha={alpha}')
axes[1].set_ylabel('Empirical FWER')
axes[1].set_title('False Positive Rate Control\n(all null; n_tests=20)', fontweight='bold')
for bar, val in zip(bars, fwer_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_05_multiple_testing.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_05_multiple_testing.png')

## Real-World Example 3: Non-Parametric Tests and Method Comparison

When data is not Gaussian (web session times, revenue -- heavy-tailed and skewed),
t-tests may have inflated Type I error or low power.

We compare: t-test vs Mann-Whitney U vs permutation test.
Permutation test makes NO distributional assumptions -- only exchangeability under H0.

In [ ]:
np.random.seed(42)

# ============================================================
# Part A: Comparison on log-normal data (skewed, heavy-tailed)
# ============================================================
n_sims  = 5000
n_group = 50
alpha_  = 0.05

# True effect: 0 (null scenario) -- measure Type I error rate
print('Type I error rate under log-normal null (should be ~5% for all):')
results_t  = []
results_mw = []
results_pm = []

for _ in range(n_sims):
    # Log-normal data: mean=exp(0+0.5)~1.65, var=large
    g1 = np.random.lognormal(0.0, 1.0, n_group)
    g2 = np.random.lognormal(0.0, 1.0, n_group)

    # t-test
    _, p_t = ttest_ind(g1, g2, equal_var=False)
    results_t.append(p_t < alpha_)

    # Mann-Whitney U
    _, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
    results_mw.append(p_mw < alpha_)

    # Permutation test (small n_perm for speed)
    obs_diff = abs(g1.mean() - g2.mean())
    combined = np.concatenate([g1, g2])
    perm_diffs = np.array([
        abs(np.random.permutation(combined)[:n_group].mean() -
            np.random.permutation(combined)[n_group:].mean())
        for _ in range(200)
    ])
    p_perm = np.mean(perm_diffs >= obs_diff)
    results_pm.append(p_perm < alpha_)

print(f'  t-test:           FP rate = {np.mean(results_t):.3f}')
print(f'  Mann-Whitney U:   FP rate = {np.mean(results_mw):.3f}')
print(f'  Permutation test: FP rate = {np.mean(results_pm):.3f}')

# True effect: 20% increase -- measure Power
print('\nPower under log-normal with +20% shift:')
results_t_pw  = []
results_mw_pw = []
results_pm_pw = []

for _ in range(1000):  # fewer for speed
    g1 = np.random.lognormal(0.0, 1.0, n_group)
    g2 = np.random.lognormal(0.2, 1.0, n_group)  # shifted mean in log-space

    _, p_t = ttest_ind(g1, g2, equal_var=False)
    results_t_pw.append(p_t < alpha_)

    _, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
    results_mw_pw.append(p_mw < alpha_)

    obs_diff = abs(g1.mean() - g2.mean())
    combined = np.concatenate([g1, g2])
    perm_diffs = np.array([abs(np.random.permutation(combined)[:n_group].mean() -
                               np.random.permutation(combined)[n_group:].mean())
                           for _ in range(200)])
    p_perm = np.mean(perm_diffs >= obs_diff)
    results_pm_pw.append(p_perm < alpha_)

print(f'  t-test:           Power = {np.mean(results_t_pw):.3f}')
print(f'  Mann-Whitney U:   Power = {np.mean(results_mw_pw):.3f}')
print(f'  Permutation test: Power = {np.mean(results_pm_pw):.3f}')

# ============================================================
# Part B: Chi-square test for categorical features
# ============================================================
print('\nChi-square test: device type vs conversion')
# Contingency table: rows=device (mobile/desktop), cols=converted (yes/no)
contingency = np.array([[150, 850],   # mobile: 150 converted, 850 not
                         [200, 800]])  # desktop: 200 converted, 800 not
chi2, p_chi2, dof, expected = chi2_contingency(contingency)
print(f'  Mobile conversion rate:  {150/1000*100:.1f}%')
print(f'  Desktop conversion rate: {200/1000*100:.1f}%')
print(f'  Chi2={chi2:.3f}, p={p_chi2:.4f}, dof={dof}')
print(f'  {'Reject H0 (device and conversion are dependent)' if p_chi2 < 0.05 else 'Fail to reject H0'}')

# ============================================================
# Part C: Method comparison table and visualization
# ============================================================
print('\nMethod comparison summary:')
methods_table = [
    ('t-test', 'Gaussian or CLT', 'Equal variance optional', 'Means', 'Fast'),
    ('Mann-Whitney U', 'No normality', 'Independent samples', 'Rank-based', 'Fast'),
    ('Permutation', 'Only exchangeability', 'Any test statistic', 'Flexible', 'Slow'),
    ('Chi-square', 'Expected counts >=5', 'Categorical data', 'Independence', 'Fast'),
]
print(f'  {'Test':<20} {'Assumption':<25} {'Requirement':<22} {'Tests':<15} {'Cost'}')
print('  ' + '-' * 90)
for row in methods_table:
    print(f'  {row[0]:<20} {row[1]:<25} {row[2]:<22} {row[3]:<15} {row[4]}')

# Final visualization: Type I error and Power bars
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

method_names = ['t-test', 'Mann-Whitney', 'Permutation']
type1_rates  = [np.mean(results_t), np.mean(results_mw), np.mean(results_pm)]
power_rates  = [np.mean(results_t_pw), np.mean(results_mw_pw), np.mean(results_pm_pw)]

x = np.arange(len(method_names))
axes[0].bar(x, type1_rates, color=['red', 'blue', 'green'], alpha=0.7)
axes[0].axhline(alpha_, color='black', ls='--', lw=2, label=f'Target alpha={alpha_}')
axes[0].set_xticks(x); axes[0].set_xticklabels(method_names)
axes[0].set_ylabel('Type I Error Rate')
axes[0].set_title('Type I Error Rate (all methods near 5%)', fontweight='bold')
for xi, val in zip(x, type1_rates):
    axes[0].text(xi, val+0.003, f'{val:.3f}', ha='center', fontsize=11)
axes[0].legend()

axes[1].bar(x, power_rates, color=['red', 'blue', 'green'], alpha=0.7)
axes[1].axhline(0.8, color='black', ls='--', lw=2, label='Target power=0.80')
axes[1].set_xticks(x); axes[1].set_xticklabels(method_names)
axes[1].set_ylabel('Statistical Power')
axes[1].set_title('Power on Log-Normal Data with 20% Shift', fontweight='bold')
for xi, val in zip(x, power_rates):
    axes[1].text(xi, val+0.01, f'{val:.3f}', ha='center', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_05_nonparametric.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_05_nonparametric.png')

print('\nKEY TAKEAWAYS for Hypothesis Testing:')
print('1. p-value = P(data this extreme | H0 true) -- NOT P(H0 is true)')
print('2. Always compute power/n BEFORE collecting data (not after)')
print('3. Multiple testing at alpha=0.05 with m=20 tests: 64% chance of FP')
print('4. Bonferroni: conservative (controls FWER); BH-FDR: less conservative')
print('5. Report effect size (Cohen d) alongside p-value for practical significance')
print('6. For skewed/non-normal data: Mann-Whitney or permutation test')
print('7. Peeking (stopping when p<0.05) inflates Type I error -- use sequential testing')

print('\nEXERCISES:')
print('1. Run 20 tests all under H0 (no real effect). Count false positives.')
print('2. Compute power if you had only n=50 per group for the A/B test.')
print('3. Apply the permutation test to the chi-square scenario.')